In [1]:
import torch
import torch.nn as nn



In [2]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('runs/vae_chunk10_new_method')

In [3]:
import numpy as np
from scipy.sparse import load_npz
datapath = '../../../data/CellCensus/3m_human_chunk_10.npz'
spare_matrix = load_npz(datapath)
dense_array = spare_matrix.toarray()

dense_tensor = torch.from_numpy(dense_array)

In [4]:
print(dense_tensor.size())

torch.Size([100000, 60664])


In [5]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, tensor):
        self.tensor = tensor

    def __getitem__(self, index):
        x = self.tensor[index]
        return x

    def __len__(self):
        return self.tensor.size(0)
       

In [6]:
from torch.utils.data import random_split, DataLoader

dataset = MyDataset(dense_tensor)

train_len = int(0.8 * len(dataset))  # 80% for training
test_len = len(dataset) - train_len  # 20% for testing


train_data, test_data = random_split(dataset, [train_len, test_len])


train_loader = DataLoader(train_data, batch_size=256, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

In [7]:
import sys
sys.path.append('../../../')
from model import VAE


device = torch.device('cuda' if torch.cuda.is_available(
) else 'mps' if torch.backends.mps.is_available() else 'cpu')

print("Use device: ", device)

input_dim = dense_tensor.size(1) 
print("input_dim: ", input_dim)


model = VAE(
    encoder= nn.Sequential(
        nn.Linear(input_dim, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
    ),
    decoder= nn.Sequential(
        nn.Linear(64, 256),
        nn.ReLU(),
        nn.Linear(256, 512),
        nn.ReLU(),
        nn.Linear(512, input_dim),
        nn.Sigmoid()
    ),
    mean = nn.Linear(256, 64),
    var = nn.Linear(256, 64)
)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

model.to(device)


Use device:  mps
input_dim:  60664
VAE model initialized


VAE(
  (encoder): Sequential(
    (0): Linear(in_features=60664, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=60664, bias=True)
    (5): Sigmoid()
  )
  (mean): Linear(in_features=256, out_features=64, bias=True)
  (var): Linear(in_features=256, out_features=64, bias=True)
)

In [8]:
def loss_function(x, x_hat, mean, log_var):
    reproduction_loss = nn.functional.mse_loss(x_hat, x, reduction='none').sum(dim=1).mean()
    KLD = torch.mean(- 0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp(), dim = 1), dim = 0)
    return reproduction_loss, KLD

In [9]:

model.train()
for i in range(100):
    overall_loss = 0
    overall_rec_loss = 0
    overall_kld_loss = 0
    for x in train_loader:
        batch_size = x.size(0)
        x = x.to(device)
        opt.zero_grad()
        _, mean, log_var, _, x_hat = model(x)
        loss_rec, loss_kld = loss_function(x, x_hat, mean, log_var)
        loss = loss_rec + loss_kld
        loss.backward()
        opt.step()
        overall_loss += (loss_rec.item() + loss_kld.item())/batch_size
        overall_rec_loss += loss_rec.item()/60664 # log the avg element-wise reconstruction loss
        overall_kld_loss += loss_kld.item()/64 # log the avg element-wise KL loss
    writer.add_scalar('Loss/overall', overall_loss/len(train_loader), i)
    writer.add_scalar('Loss/rec', overall_rec_loss/len(train_loader), i)
    writer.add_scalar('Loss/kld', overall_kld_loss/len(train_loader), i)
    print(f"Epoch {i}, Loss: {overall_loss/len(train_loader)}, Rec: {overall_rec_loss/len(train_loader)}, KLD: {overall_kld_loss/len(train_loader)}")

model.eval()

with torch.no_grad():
    rec_loss_eval = 0
    kld_loss_eval = 0
    for x in test_loader:
        x = x.to(device)
        _, _, _, _, x_hat = model(x)
        loss_rec = nn.functional.mse_loss(x_hat, x, reduction='mean')
        rec_loss_eval += loss_rec.item()
    writer.add_scalar('Loss/rec_eval', rec_loss_eval/len(test_loader), i)
    print(f"Loss: {rec_loss_eval/len(test_loader)}")

Epoch 0, Loss: 17.827837359602785, Rec: 0.0740379137168033, KLD: 0.92113109100491
Epoch 1, Loss: 16.09542614973772, Rec: 0.0672688450458406, KLD: 0.4146327357322644
Epoch 2, Loss: 15.809095816347545, Rec: 0.06602744340088675, KLD: 0.4342030100167369
Epoch 3, Loss: 15.646599833624432, Rec: 0.0653864392925551, KLD: 0.4123418761518436
Epoch 4, Loss: 15.548092912537411, Rec: 0.06497348066911436, KLD: 0.3884790015106384
Epoch 5, Loss: 15.474361060383602, Rec: 0.06471092214024557, KLD: 0.3565217566947206
Epoch 6, Loss: 15.422668446033907, Rec: 0.0645255380355026, KLD: 0.33924187039034054
Epoch 7, Loss: 15.383618651416164, Rec: 0.06437712270039656, KLD: 0.3326994497745563
Epoch 8, Loss: 15.37189393226331, Rec: 0.06428487234357093, KLD: 0.33108152901402677
Epoch 9, Loss: 15.346840222160846, Rec: 0.06418157464143934, KLD: 0.3323514705267958
Epoch 10, Loss: 15.323524681714396, Rec: 0.06409146644854419, KLD: 0.33326959895630615
Epoch 11, Loss: 15.293800779805778, Rec: 0.06400145414868108, KLD: 0.

In [10]:
# import os
# import time 
# date = time.strftime("%Y%m%d-%H%M%S")
# current = os.getcwd()
# os.makedirs(f"./trained_model/vae_{date}", exist_ok=True)
# torch.save(model, f"./trained_model/vae_{date}/vae_{date}.pt")
# torch.save(model.state_dict(),
#            f"./trained_model/vae_{date}/vae_{date}_state_dict.pt")
# torch.save(opt.state_dict(),
#            f"./trained_model/vae_{date}/optimizer_{date}_state_dict.pt")

# writer.close()